# 05 - Measure what changed in Brief A

**Purpose:** explain the change in test volume between H1 2025 and H1 2026.

The selected method is the simple growth tree:

`test lines = people ? visits per person ? test lines per visit`

The other sections are reference examples only. The current Brief-A result is the Hypo A recalculation in `data/hypo_A/brief_a_iitgn_metric_summary.csv` and `docs/brief-a-iitgn-aligned-results.md`.


## CONFIG

In [ ]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "A",
    "group": "6",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-07-01"),
    "period_last": ("2025-01-01", "2025-07-01"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

bk = pd.read_csv(CONFIG["clean"] / "bookings_clean.csv.gz", parse_dates=["slot_at", "collected_at"], dtype={"booking_id": "string", "person_id": "string", "rescheduled_from": "string", "status_at_export": "string"})
last = (bk["slot_at"] >= pd.Timestamp(CONFIG["period_last"][0])) & (bk["slot_at"] < pd.Timestamp(CONFIG["period_last"][1]))
this = (bk["slot_at"] >= pd.Timestamp(CONFIG["period_this"][0])) & (bk["slot_at"] < pd.Timestamp(CONFIG["period_this"][1]))
bk["half"] = np.select([last, this], ["H1_2025", "H1_2026"], default="other")
bk = bk[bk["half"] != "other"]
print(bk.groupby("half").agg(bookings=("booking_id", "size"), tests=("n_tests", "sum"), people=("person_id", "nunique")))

## 1  The Brief-A growth tree

Tests = people  visits per person  tests per visit. Growth in a product is close to the sum of growths in the factors (exactly so in logs). Compute each factor for both halves and check that they multiply back.

In [ ]:
def tree(df, by=None):
    g = df.groupby(([by] if by else []) + ["half"], observed=True).agg(tests=("n_tests", "sum"), visits=("booking_id", "size"), people=("person_id", "nunique"))
    g["visits_per_person"] = g["visits"] / g["people"]
    g["tests_per_visit"] = g["tests"] / g["visits"]
    return g

t = tree(bk)
growth = (t.xs("H1_2026") / t.xs("H1_2025") - 1) * 100
print(growth.round(2))
# log-decomposition: the branch growths sum to the total growth exactly
logs_ = np.log(t.xs("H1_2026") / t.xs("H1_2025"))
checks.check("branches multiply back to the total (log decomposition)", abs(logs_["people"] + logs_["visits_per_person"] + logs_["tests_per_visit"] - logs_["tests"]) < 1e-9,
             f"people {logs_['people']:+.4f} + visits/person {logs_['visits_per_person']:+.4f} + tests/visit {logs_['tests_per_visit']:+.4f} = {logs_['tests']:+.4f}")

In [ ]:
by_city = tree(bk, "city")
city_growth = ((by_city.xs("H1_2026", level="half") / by_city.xs("H1_2025", level="half") - 1) * 100).round(1)
city_growth[["tests", "people", "visits_per_person", "tests_per_visit"]].sort_values("tests")

> **Interview angle.** Test volume changed between H1 2025 and H1 2026. Explain the change using people, visits per person and test lines per visit, then state what you would verify before presenting it.
>
> *A simple answer should say:* Roughly 6 points sit in visits per patient and revenue per visit together; log growth makes the split exact. Check that the patient count is a count of people and not of ids (duplicates), that both periods use the same definition of a visit, and that a system change did not move the grain.

## 2 - Save the Brief-A chart

One chart, the claim's number against its comparison, with the denominator in the axis label. Save it to `slides/`.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.2))
g = city_growth["tests"].sort_values()
ax.barh(g.index, g.values, color=["#D63A6A" if v < 0 else "#4E38AB" for v in g.values])
ax.axvline(0, color="#1A1440", lw=1)
ax.set_xlabel("Tests, H1 2026 vs H1 2025, percent change (denominator: tests on collected and no-show bookings, H1 2025)")
ax.set_title("Where the growth went, by city")
for i, v in enumerate(g.values): ax.text(v + (0.3 if v >= 0 else -0.3), i, f"{v:+.1f}%", va="center", ha="left" if v >= 0 else "right", fontsize=9)
plt.tight_layout()
(ROOT / "slides").mkdir(exist_ok=True)
fig.savefig(ROOT / "slides" / "slide4_chart.png", dpi=200)
print("saved slides/slide4_chart.png")
checks.summary()

## Before continuing

- Which number is on slide 3, and what are its denominator and period? Say them aloud.
- Which of the five methods did you use, and why not the nearest other one? That is an approach-log row.
- What would make you wrong? That is slide 5.